# 01 — Text Classification Baseline: TF-IDF + Logistic Regression

Companion notebook to `01-text-classification-fundamentals.md`.

We build the baseline you'd build **before** reaching for BERT: a TF-IDF vectorizer feeding a
Logistic Regression classifier, trained on a small hand-written synthetic dataset of pharma-style
marketing "claim" sentences labeled by claim type (`efficacy`, `safety`, `dosing`, `comparative`).

This mirrors the real workflow — establish a fast, interpretable baseline first, and only justify a
fine-tuned transformer (notebook 02) if it clearly beats this on held-out data.

Runs fully offline, CPU only, in well under a minute.


In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Synthetic claims dataset

Hand-written example sentences in the style of pharma marketing claims, each labeled with a single
claim-type category. In production this would be multi-label (chapter 01 explains why), but for a
clean baseline demo we keep it single-label multi-class here.


In [2]:
data = [
    # efficacy claims
    ("Drug X reduces symptom severity by 42% compared to placebo.", "efficacy"),
    ("Patients taking Drug X experienced significantly improved outcomes.", "efficacy"),
    ("Clinical trials demonstrate a 35% reduction in relapse rates.", "efficacy"),
    ("Drug X lowers the frequency of flare-ups versus standard of care.", "efficacy"),
    ("In a 12-week study, symptom scores decreased substantially with Drug X.", "efficacy"),
    ("Drug X improves lung function scores within the first month of treatment.", "efficacy"),
    ("A majority of patients achieved remission while on Drug X.", "efficacy"),
    ("Drug X was shown to decrease disease progression by nearly half.", "efficacy"),
    ("Symptom-free days increased significantly with Drug X therapy.", "efficacy"),
    ("Drug X demonstrated superior response rates in the pivotal trial.", "efficacy"),
    # safety claims
    ("The most common adverse reaction was mild headache.", "safety"),
    ("Drug X carries a boxed warning for increased cardiovascular risk.", "safety"),
    ("Serious allergic reactions have been reported with Drug X.", "safety"),
    ("Drug X should not be used in patients with severe liver impairment.", "safety"),
    ("Common side effects include nausea, fatigue, and dizziness.", "safety"),
    ("Drug X has not been shown to increase the risk of infection.", "safety"),
    ("Discontinue Drug X immediately if signs of an allergic reaction occur.", "safety"),
    ("Long-term safety data show no significant increase in adverse events.", "safety"),
    ("Drug X is contraindicated in patients with a known hypersensitivity.", "safety"),
    ("Monitor liver function periodically while taking Drug X.", "safety"),
    # dosing claims
    ("The recommended starting dose of Drug X is 10mg once daily.", "dosing"),
    ("Drug X should be taken with food to improve absorption.", "dosing"),
    ("Dose adjustment is recommended for patients with renal impairment.", "dosing"),
    ("The maximum daily dose of Drug X is 40mg.", "dosing"),
    ("Drug X is available in 5mg, 10mg, and 20mg tablets.", "dosing"),
    ("Missed doses of Drug X should not be doubled up.", "dosing"),
    ("Drug X dosing may be titrated upward after two weeks.", "dosing"),
    ("Administer Drug X subcutaneously once every four weeks.", "dosing"),
    ("No dose adjustment is required for elderly patients.", "dosing"),
    ("Drug X should be discontinued gradually, not stopped abruptly.", "dosing"),
    # comparative claims
    ("Drug X is more effective than Drug Y at reducing flare frequency.", "comparative"),
    ("In head-to-head trials, Drug X outperformed the standard-of-care therapy.", "comparative"),
    ("Drug X showed a faster onset of action compared to Drug Z.", "comparative"),
    ("Unlike Drug Y, Drug X does not require weekly blood monitoring.", "comparative"),
    ("Drug X achieved higher remission rates than the leading competitor.", "comparative"),
    ("Patients preferred Drug X over Drug Y in a head-to-head satisfaction survey.", "comparative"),
    ("Drug X has a more favorable dosing schedule than comparable therapies.", "comparative"),
    ("Drug X was superior to placebo and non-inferior to Drug Y.", "comparative"),
    ("Drug X demonstrated a better safety profile than older-generation therapies.", "comparative"),
    ("Compared to Drug Y, Drug X reduced hospitalization rates by 20%.", "comparative"),
]

df = pd.DataFrame(data, columns=["text", "label"])
print(f"Dataset size: {len(df)} examples across {df['label'].nunique()} classes")
df["label"].value_counts()


Dataset size: 40 examples across 4 classes


label
efficacy       10
safety         10
dosing         10
comparative    10
Name: count, dtype: int64

## 2. Train/test split, TF-IDF vectorization

In [3]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.3, random_state=RANDOM_STATE, stratify=df["label"],
)

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")


Train size: (28, 258), Test size: (12, 258)
Vocabulary size: 258


## 3. Train Logistic Regression and evaluate

In [4]:
clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.3f}")
print(f"Macro F1: {macro_f1:.3f}\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.750
Macro F1: 0.760

              precision    recall  f1-score   support

 comparative       0.50      0.67      0.57         3
      dosing       1.00      0.67      0.80         3
    efficacy       0.67      0.67      0.67         3
      safety       1.00      1.00      1.00         3

    accuracy                           0.75        12
   macro avg       0.79      0.75      0.76        12
weighted avg       0.79      0.75      0.76        12



## 4. Interpretability: which words drive each class?

One of the practical advantages of a linear model over TF-IDF: you can directly inspect which
n-grams push a prediction toward a given claim type — useful when a reviewer asks "why did the
model tag this as a safety claim?"


In [5]:
feature_names = np.array(vectorizer.get_feature_names_out())

for i, class_label in enumerate(clf.classes_):
    top_idx = np.argsort(clf.coef_[i])[-8:][::-1]
    top_terms = feature_names[top_idx]
    print(f"{class_label:12s} -> {', '.join(top_terms)}")


comparative  -> head, drug, compared drug, drug drug, head head, compared, superior placebo, non inferior
dosing       -> dose, daily, dose drug, 10mg, subcutaneously weeks, weeks, administer drug, administer
efficacy     -> symptom, rates, remission drug, patients achieved, remission, achieved remission, achieved, majority patients
safety       -> allergic, reaction, adverse, reported, allergic reactions, reported drug, reactions, reactions reported


## 5. Try it on a new, unseen claim sentence


In [6]:
new_sentences = [
    "Drug X reduced disease flare-ups by over a third versus placebo.",
    "Patients should take Drug X once daily in the morning with water.",
    "Drug X was associated with a low rate of serious adverse events.",
    "Drug X outperformed the previous standard treatment in a direct comparison.",
]

new_vecs = vectorizer.transform(new_sentences)
predictions = clf.predict(new_vecs)
probabilities = clf.predict_proba(new_vecs)

for sent, pred, probs in zip(new_sentences, predictions, probabilities):
    confidence = probs.max()
    print(f"[{pred:11s} | conf={confidence:.2f}] {sent}")


[efficacy    | conf=0.32] Drug X reduced disease flare-ups by over a third versus placebo.
[dosing      | conf=0.31] Patients should take Drug X once daily in the morning with water.
[safety      | conf=0.34] Drug X was associated with a low rate of serious adverse events.
[comparative | conf=0.29] Drug X outperformed the previous standard treatment in a direct comparison.


## Takeaways

- TF-IDF + Logistic Regression is fast to train, fully interpretable, and a legitimate baseline —
  not a toy. On this small synthetic dataset it separates the four claim types cleanly because the
  vocabulary is fairly distinctive per class.
- What this baseline **can't** do: understand negation ("Drug X does **not** increase risk" vs.
  "Drug X increases risk" share almost the same bag-of-words), or generalize to paraphrases using
  vocabulary never seen in training. That gap is exactly what motivates the transfer-learning /
  BERT fine-tuning approach in notebook `02_bert_finetuning_demo.ipynb` and chapter
  `02-transfer-learning-and-bert-finetuning.md`.
- In a real multi-label setting (a claim can be both `efficacy` and `safety`), swap
  `LogisticRegression` for `OneVsRestClassifier(LogisticRegression())` or a native multi-label
  formulation, and evaluate with per-label precision/recall/F1 rather than single-label accuracy.
